In [2]:
!pip install requests pandas

import requests
import pandas as pd

In [11]:
import pandas as pd
import folium
from folium.plugins import HeatMap
from google.colab import files
import time

API_KEY = '97bf90f8-afca-4468-ac25-43a3c1246b40' # зарегался на "2гис: менеджер платформы" и создал api-ключ

# 1. ищем id Москвы
region_url = "https://catalog.api.2gis.com/2.0/region/search"
region_params = {'q': 'Москва', 'key': API_KEY}
region_response = requests.get(region_url, params=region_params)
region_response.raise_for_status() # проверка на ошибки
region_data = region_response.json() # json --> словарь
print(region_data)
moscow_id = None
if region_data.get('result') and region_data['result'].get('items'):
    moscow_id = region_data['result']['items'][0]['id']
    print(f"id Москвы: {moscow_id}")
else:
    print("не нашли id Москвы")

1. ищем id Москвы
{'meta': {'api_version': '2.0.20462', 'code': 200, 'issue_date': '20260513'}, 'result': {'items': [{'id': '32', 'name': 'Москва', 'type': 'region'}], 'total': 1}}
id Москвы: 32


In [4]:
# 2. ищем автосалоны в Москве (5 страниц по 10 записей)
search_url = "https://catalog.api.2gis.com/3.0/items"

all_items = [] # резы

# цикл для 5 страниц
for current_page in range(1, 2):
    print(f"Загружаю страницу {current_page}...")

    search_params = {
        'q': 'автосалон',
        'region_id': moscow_id,
        'key': API_KEY,
        'fields': 'items.point,items.address_name',
        'page_size': 10,    # по 10 записей на стр.
        'page': current_page
    }

    search_response = requests.get(search_url, params=search_params)
    search_response.raise_for_status() # проверка на ошибки
    search_data = search_response.json() # json --> словарь
    print(search_data)

    # берём данные со страницы
    items = search_data.get('result', {}).get('items', []) # если не нашли, то пустой словарь/список
    if not items:
        print(f"На странице {current_page} данные закончились, прерываю загрузку.")
        break

    all_items.extend(items)  # .extend в отличие от .append распакует список перед добавлением
    print(f"Найдено {len(items)} салонов. Всего на сейчас: {len(all_items)}")

    time.sleep(0.3)  # пауза между запросами

print(f"\nВсего загружено объектов: {len(all_items)}")

2. ищем автосалоны в Москве (5 страниц по 10 записей)
Загружаю страницу 1...
{'meta': {'api_version': '3.0.20462', 'code': 200, 'issue_date': '20260513'}, 'result': {'items': [{'address_name': 'Калужское шоссе 21 километр, 3', 'id': '70000001080929874', 'name': 'Fresh Москва, автомобильный маркетплейс', 'point': {'lat': 55.602318, 'lon': 37.486022}, 'type': 'branch'}, {'address_name': 'МКАД 40 километр, вл1 ст3', 'id': '70000001065399079', 'name': 'Рольф Ясенево, автоцентр', 'point': {'lat': 55.59764, 'lon': 37.506308}, 'type': 'branch'}, {'address_name': 'МКАД 44 километр, вл1 ст2', 'id': '70000001089638586', 'name': 'Peleton, автомолл по продаже авто с пробегом', 'point': {'lat': 55.628677, 'lon': 37.470255}, 'type': 'branch'}, {'address_name': 'Варшавское шоссе, 125 ст1в', 'full_name': 'Москва, Варшавское шоссе, 125 ст1в', 'id': '70000001045242956', 'name': 'Auto Expert, автосалон автомобилей с пробегом', 'point': {'lat': 55.626443, 'lon': 37.618232}, 'purpose_name': 'Автосалон', 't

In [9]:
# 3. обработка данных + табличка
print(all_items)
salons = []
for item in all_items:
    name = item.get('name')
    address = item.get('address_name')
    point = item.get('point')

    if point and isinstance(point, dict):  # проверяем: point непустой словарь
        lat = point.get('lat')
        lon = point.get('lon')
    else:
        lat, lon = None, None

    salons.append({'Название': name, 'Адрес': address, 'Широта': lat, 'Долгота': lon})

df = pd.DataFrame(salons)
print(f"Создана таблица из {len(df)} записей")
print("\nПервые 5 строк:")
print(df.head())

[{'address_name': 'Калужское шоссе 21 километр, 3', 'id': '70000001080929874', 'name': 'Fresh Москва, автомобильный маркетплейс', 'point': {'lat': 55.602318, 'lon': 37.486022}, 'type': 'branch'}, {'address_name': 'МКАД 40 километр, вл1 ст3', 'id': '70000001065399079', 'name': 'Рольф Ясенево, автоцентр', 'point': {'lat': 55.59764, 'lon': 37.506308}, 'type': 'branch'}, {'address_name': 'МКАД 44 километр, вл1 ст2', 'id': '70000001089638586', 'name': 'Peleton, автомолл по продаже авто с пробегом', 'point': {'lat': 55.628677, 'lon': 37.470255}, 'type': 'branch'}, {'address_name': 'Варшавское шоссе, 125 ст1в', 'full_name': 'Москва, Варшавское шоссе, 125 ст1в', 'id': '70000001045242956', 'name': 'Auto Expert, автосалон автомобилей с пробегом', 'point': {'lat': 55.626443, 'lon': 37.618232}, 'purpose_name': 'Автосалон', 'type': 'branch'}, {'address_comment': 'цокольный этаж', 'address_name': 'Ленинградское шоссе, вл21', 'id': '70000001086696598', 'name': 'Рольф Химки, автосалон автомобилей с пр

In [15]:
# 4. карта

df_clean = df.dropna(subset=['Широта', 'Долгота']).copy()
print(f"Записей с координатами: {len(df_clean)}")

if not df_clean.empty:
    center_lat = df_clean['Широта'].mean()
    center_lon = df_clean['Долгота'].mean()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=11) # центр карты (location) + масштаб (zoom_start)

    for _, row in df_clean.iterrows(): # .iterrows() возвращает пары (индекс, строка) для каждой строки df
        folium.Marker(
            location=[row['Широта'], row['Долгота']],
            popup=f"{row['Название']}<br>{row['Адрес']}",
            icon=folium.Icon(color='red', icon='car', prefix='fa')
        ).add_to(m)

    heat_data = [[row['Широта'], row['Долгота']] for _, row in df_clean.iterrows()]
    print(heat_data)
    HeatMap(heat_data, radius=12, blur=10).add_to(m)

    map_file = 'autosalon_map_50.html'
    m.save(map_file)
    files.download(map_file)
    print(f"\nКарта '{map_file}' сохранена и скачана. Откройте её.")
else:
    print("Нет данных для построения карты.")

Записей с координатами: 10
<generator object DataFrame.iterrows at 0x79090a9d1f10>
[[55.602318, 37.486022], [55.59764, 37.506308], [55.628677, 37.470255], [55.626443, 37.618232], [55.905949, 37.412602], [55.862057, 37.582294], [55.759458, 37.53509], [55.85805, 37.396015], [55.650922, 37.441674], [55.896444, 37.427962]]
